<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Set up the speech project

**[Set up the speech project](https://learning.nextia-ai.com/courses/speech/m01/set-up-the-speech-project/)** · Speech AI: Transcribe, Understand and Speak · Module 1, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** get the Larkfield calls, look at one call as numbers, listen to it with its reference transcript, see what a phone line removes, and transcribe the call with a small Whisper model on your own computer.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 80 MB with the Whisper model). No account, no key and no graphics card. |
| **You should know** | Sample rate, channels and spectrograms, from [Audio as data](https://learning.nextia-ai.com/courses/speech/m01/audio-as-data/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m01/set-up-the-speech-project/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M01-L03-set-up-the-speech-project/set-up-the-speech-project-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, faster-whisper 1.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. faster-whisper brings its own engine (CTranslate2) and the Silero voice detector: the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'faster-whisper']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 5 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/calls/wideband/C-0225.flac": "cda4ffafd9f3a64c375f8c3dee340f2a6ea3f6dd9a6023bb6aa98788b36dcd4a",
    "data/calls/phone/C-0225.wav": "0eab5a465ce8234b1c6e8354f7447cdaa5434eef6d7de6ccfb09ad0978d699b0",
    "data/calls/phone/C-0083.wav": "8896c2f5abf5d1fc71248906bfd6341e778efc4bf6acf24cd0115e000ef90e24",
    "data/calls/phone/C-0089.wav": "00c33ee1c3d2bcf4639c465df8e945729b382c9ace180e986bc3ac4714b7e048",
    "data/calls/phone/C-0219.wav": "cf505a17cbcd93cfeacfedd50dd424be352bf42676461b2b174124345b8e6172",
    "data/calls/phone/C-0102.wav": "021d363cdff36def15c639cbecb8e6c45d1dc7c52bdb506721ca1d4cd6a11d54",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One call as numbers

Priya starts with one call: C-0225, a customer who asks how to set up a Cedar raised bed. The call set has two versions of each call:

- `calls/phone/`: what a phone system stores. 8,000 samples a second, G.711 mu-law (8 bits a sample). Stereo calls have the agent on channel 1 and the caller on channel 2.
- `calls/wideband/`: the same call at 16,000 samples a second, as an HD voice line carries it.

Run the next cell. It reads the facts of both files and the loudness of the phone call.

In [ ]:
calls = load_calls()
call = calls["C-0225"]
for path in ("data/calls/phone/C-0225.wav", "data/calls/wideband/C-0225.flac"):
    i = sf.info(path)
    print(f"{path}: {i.samplerate} samples a second, {i.channels} channel(s), {i.duration:.1f} s, {i.subtype}")
phone, sr = read_audio("data/calls/phone/C-0225.wav")
print(f"loudness of the phone call: {dbfs(phone):.1f} dBFS")

> **Check.** The phone file has 8000 samples a second and 2 channels; the wideband file has 16000 and 1 channel. Both are 23.3 seconds long. The loudness is about -30.0 dBFS (decibels below full scale: 0 is the loudest possible sound).

A call also has quiet parts: while one person listens, and between turns. The next cell counts the 30 ms pieces (frames) that are quieter than -45 dBFS.

In [ ]:
def silence_share(x, sr, frame=0.03, threshold_db=-45):
    """The share of 30 ms frames that are quieter than threshold_db."""
    n = int(sr * frame)
    frames = x[: len(x) // n * n].reshape(-1, n)
    levels = 20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)
    return float(np.mean(levels < threshold_db))

print(f"share of silence in the phone call: {silence_share(phone, sr):.2f}")

> **Check.** You should see about 0.42: that share of the call is silence.

## 2. Listen to the call

Run the next cell. It shows a player (press the play button; it does not start by itself) and prints the reference transcript: what each person said, with the time of each turn. The voices are synthetic: Piper and Kokoro voices read our script.

In [ ]:
stereo, sr = sf.read("data/calls/phone/C-0225.wav", dtype="float32")
display(Audio(stereo.T, rate=sr))  # stereo: the agent on the left, the caller on the right
for turn in call["turns"]:
    print(f'{turn["start"]:5.1f}-{turn["end"]:5.1f} s  {turn["speaker"]:6}  {turn["text"]}')

## 3. What a phone line removes

A phone line keeps only the sounds between about 300 and 3,400 Hz. The wideband version keeps sounds up to 8,000 Hz. The difference is easy to see in a **spectrogram**: time goes from left to right, pitch goes up, and a bright colour means a loud sound at that pitch.

> **Predict.** The agent says "Thank you for calling Larkfield, this is Chloe." Which spectrogram has bright parts above the dashed line at 3,400 Hz: the phone one, the wideband one, or both?

In [ ]:
turn = call["turns"][0]                                 # the agent's greeting
n = len(turn["text"].split(". ")[0].split())             # the words of its first sentence
start, end = turn["words"][0]["start"], turn["words"][n - 1]["end"]
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for col, (name, path, channel) in enumerate([("phone, 8 kHz", "data/calls/phone/C-0225.wav", 0),
                                             ("wideband, 16 kHz", "data/calls/wideband/C-0225.flac", None)]):
    x, sr = read_audio(path, channel=channel)
    piece = x[int(start * sr):int(end * sr)]
    axes[0, col].plot(np.arange(len(piece)) / sr, piece, lw=0.4)
    axes[0, col].set_title(f"waveform, {name}")
    f, t, power = signal.spectrogram(piece, fs=sr, nperseg=int(0.032 * sr), noverlap=int(0.024 * sr))
    db = 10 * np.log10(power + 1e-12)
    axes[1, col].pcolormesh(t, f, db, shading="auto", vmin=db.max() - 70, vmax=db.max())
    axes[1, col].axhline(3400, color="white", ls="--", lw=1)
    axes[1, col].set_ylim(0, 8000); axes[1, col].set_title(f"spectrogram, {name}")
    axes[1, col].set_xlabel("seconds"); axes[1, col].set_ylabel("Hz")
plt.tight_layout(); plt.show()

> **Check.** Only the wideband spectrogram has sound above 3,400 Hz: the "s" and "f" sounds live there. The phone spectrogram stops at 4,000 Hz (half of 8,000 samples a second), and the line removed most sound above 3,400 Hz. This is one reason why recognisers make more mistakes on phone calls.

## 4. Transcribe the call with Whisper

Whisper is an open speech recogniser from OpenAI (MIT licence). faster-whisper runs it fast on a CPU. The next cell downloads the smallest size, `tiny` (about 75 MB), at a fixed version, and loads it.

In [ ]:
import time
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download

model_path = snapshot_download("Systran/faster-whisper-tiny", revision="d90ca5fe260221311c53c58e660288d3deb8d356")
model = WhisperModel(model_path, device="cpu", compute_type="int8", cpu_threads=4)
print("Whisper tiny is ready.")

Whisper expects 16,000 samples a second, so the next cell first changes the phone call's rate from 8,000 to 16,000 (`read_audio(..., sr=16000)`). Changing the rate adds samples, never information: the sounds that the phone line removed stay lost.

Then it transcribes the call, times it, and compares the result with the reference after the course's **normalisation guide** (lower case, no punctuation, numbers as digits; Module 2 explains it). The **word error rate** (WER) is the share of reference words that the model got wrong. The **real-time factor** (RTF) is the compute time divided by the length of the audio: 0.1 means one hour of audio in six minutes.

In [ ]:
x16, _ = read_audio("data/calls/phone/C-0225.wav", sr=16000)   # both channels mixed, 16 kHz
start = time.perf_counter()
segments, _ = model.transcribe(x16, language="en", beam_size=1, condition_on_previous_text=False)
text = " ".join(s.text.strip() for s in segments)
compute = time.perf_counter() - start

ref = reference(call)
print("Whisper tiny:", text, "\n")
wer_tiny = cp.wer(cp.normalise(ref, "full", "en"), cp.normalise(text, "full", "en"))
print(f"WER after the guide: {wer_tiny:.3f}")
print(f"compute {compute:.2f} s for {len(x16) / 16000:.1f} s of audio: RTF {cp.rtf(compute, len(x16) / 16000):.3f}")

> **Check.** You should see a WER of about 0.055. The course's recorded run of the same model on this call gave 0.055, in 0.35 seconds (RTF 0.015 on an Apple M4 laptop with 4 threads). Your time depends on your computer; the first run is a little slower. On a free Colab or Kaggle CPU, expect a few times slower.

Look at the words that differ. Most mistakes on a phone call are small words and names.

## 5. Change one thing: no normalisation

The guide decides what counts as the same word. Run the next cell: it compares the raw texts, with capital letters and punctuation.

> **Predict.** Without the guide, does the WER go up or down?

In [ ]:
raw = cp.wer(cp.normalise(ref, "none"), cp.normalise(text, "none"))
print(f"WER without the guide: {raw:.3f}   with the guide: {wer_tiny:.3f}")

> **Check.** About 0.122 without the guide: the same transcript looks much worse, because a word with a capital letter or a comma counts as a different word ("bed," is not "bed"). Always compare models after the same normalisation; Module 2 makes this the rule for the whole course.

## 6. Failure case: the wrong sample rate

A common mistake: you give Whisper the 8 kHz samples as they are. Whisper thinks that they are 16 kHz audio, so for the model the call plays twice as fast and an octave higher.

In [ ]:
x8, _ = read_audio("data/calls/phone/C-0225.wav")              # 8 kHz: the wrong rate for Whisper
segments, _ = model.transcribe(x8, language="en", beam_size=1, condition_on_previous_text=False)
wrong = " ".join(s.text.strip() for s in segments)
print(wrong[:300])
print(f"WER: {cp.wer(cp.normalise(ref, 'full', 'en'), cp.normalise(wrong, 'full', 'en')):.3f}")

> **Check.** The transcript is wrong (the model hears fast, high speech and invents sentences), and the WER is many times higher. There is no error message: the code runs, and the result is bad. **Fix:** always convert to the rate the model expects (`read_audio(path, sr=16000)`), and check the rate with `sf.info` first.

## 7. Your turn: inspect five calls

> **Your turn.** For the phone files of the five calls C-0225, C-0083, C-0089, C-0219 and C-0102, make a list `table` with one dictionary for each call (in this order) with: `call_id`, `rate` and `channels` (from `sf.info`), `seconds` (`info.duration`, rounded to 1 decimal), `loudness` (`dbfs` of the audio with both channels mixed, rounded to 1 decimal) and `silence` (`silence_share`, rounded to 2 decimals).

In [ ]:
table = []
for call_id in ["C-0225", "C-0083", "C-0089", "C-0219", "C-0102"]:
    path = f"data/calls/phone/{call_id}.wav"
    info = sf.info(path)
    x, sr = read_audio(path)
    table.append({"call_id": call_id, "rate": info.samplerate, "channels": info.channels,
                  "seconds": round(info.duration, 1), "loudness": round(dbfs(x), 1),
                  "silence": round(silence_share(x, sr), 2)})
for row in table:
    print(row)

Run the three checks.

In [ ]:
expect_hash('the rates and channels', None if table is ... else [(r["call_id"], r["rate"], r["channels"]) for r in table], '07ebd223ccd81a2f')

In [ ]:
expect("the call with the largest share of silence",
       None if table is ... else max(table, key=lambda r: r["silence"])["call_id"], 'C-0219')

In [ ]:
expect("the mean loudness of the five calls",
       None if table is ... else round(sum(r["loudness"] for r in table) / len(table), 1), -25.9, tolerance=0.004)

> **Check.** Compare the calls. A mono call mixes both people into one channel. Which call has the most silence, and why? Look at its labels: `calls[call_id]["planted"]` lists the practice problems we made on purpose in a call.

## Reset

If you change files in `larkfield-calls` and want to start again, delete the folder and run the setup cells again. In the course project on your computer, `speech reset` checks every data file against its checksum.

## Next

You opened the Larkfield calls, listened to one, saw what the phone line removes and transcribed it with Whisper. The [lesson page](https://learning.nextia-ai.com/courses/speech/m01/set-up-the-speech-project/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m01/set-up-the-speech-project/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.